## Import package

In [3]:
import pandas as pd
import numpy as np

In [4]:
pd.set_option('display.max_columns',150)
pd.set_option('display.max_rows',100)

## Import Risk Segment PD Output

In [5]:
risk_df=pd.read_pickle('../data/risk_segmented_output.pkl')
risk_df.columns.to_list()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'purpose',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'total_rec_late_fee',
 'last_pymnt_d',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op_rev_tl',
 'num_rev_accts',

In [6]:
risk_df[['actual_default_flag','pd_score','risk_band','risk_score']].tail(10)

,actual_default_flag,pd_score,risk_band,risk_score
692153,0,0.468976,Medium,531.0
1816200,0,0.233568,Low,766.0
1270714,0,0.050490,Very Low,950.0
1709748,0,0.533128,High,467.0
596968,0,0.739552,Very High,260.0
1135476,1,0.222382,Very Low,778.0
1601713,1,0.950497,Very High,50.0
344609,0,0.146255,Very Low,854.0
874428,1,0.554675,High,445.0
1910437,1,0.368561,Medium,631.0


In [7]:
risk_df.shape

(40000, 124)

In [8]:
full_df=pd.read_pickle('../data/lending_club_feature_engineered.pkl')
full_df.shape

(1340973, 128)

In [9]:
full_df.columns.to_list()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'purpose',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 

## Recovery columns for LGD 

In [14]:
recovery_cols = [
    "funded_amnt",
    "funded_amnt_clean",
    "total_rec_prncp",
    "recoveries",
    "collection_recovery_fee",
    "int_rate",
    "grade",
    "purpose"
]

available_recovery_cols = [cols for cols in recovery_cols if cols in full_df.columns]
print(available_recovery_cols)

risk_df = risk_df.join(full_df[available_recovery_cols], rsuffix="_orig")
print(risk_df.shape)

['funded_amnt', 'funded_amnt_clean', 'total_rec_prncp', 'recoveries', 'collection_recovery_fee', 'int_rate', 'grade', 'purpose']
(40000, 140)


In [15]:
risk_df[available_recovery_cols].head()

,funded_amnt,funded_amnt_clean,total_rec_prncp,recoveries,collection_recovery_fee,int_rate,grade,purpose
824713,12000,12000,5138.16,0.0,0.0,19.89,E,debt_consolidation
591446,20100,20100,20100.00,0.0,0.0,15.59,C,debt_consolidation
1789024,8000,8000,8000.00,0.0,0.0,11.14,B,credit_card
1084390,23325,23325,23325.00,0.0,0.0,12.69,C,credit_card
1807296,21250,21250,21250.00,0.0,0.0,22.95,F,debt_consolidation


## Creating EAD (Exposure At Default)

In [16]:
risk_df['ead'] = risk_df['funded_amnt_clean']

risk_df['ead'] = risk_df['ead'].fillna(risk_df['funded_amnt'])
risk_df['ead'] = risk_df['ead'].replace(0, np.nan)

risk_df['ead'].describe()

count    40000.000000
mean     14513.557500
std       8711.779018
min        500.000000
25%       8000.000000
50%      12000.000000
75%      20000.000000
max      40000.000000
Name: ead, dtype: float64

## Creating Realized Loss

In [18]:
risk_df['realized_loss'] = risk_df['ead'] - risk_df['total_rec_prncp'].fillna(0) - risk_df['recoveries'].fillna(0)

risk_df['realized_loss'] = risk_df['realized_loss'].clip(lower=0)

risk_df[['ead', 'total_rec_prncp', 'recoveries', 'realized_loss']].head()

,ead,total_rec_prncp,recoveries,realized_loss
824713,12000,5138.16,0.0,6861.84
591446,20100,20100.00,0.0,0.00
1789024,8000,8000.00,0.0,0.00
1084390,23325,23325.00,0.0,0.00
1807296,21250,21250.00,0.0,0.00


## Creating Realized LGD

In [21]:
risk_df['realized_lgd'] = risk_df['realized_loss']/risk_df['ead'] \
    .replace([np.inf,-np.inf], np.nan)\
    .fillna(0)\
    .clip(lower=0, upper=1)

risk_df['realized_lgd'].describe()

count    40000.000000
mean      2229.085226
std       5400.007656
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      40000.000000
Name: realized_lgd, dtype: float64

## Estimate LGD Assumption from Defaulted Borrowers

In [23]:
defaulted_loans = risk_df[risk_df['actual_default_flag'] == 1].copy()

global_lgd = defaulted_loans['realized_lgd'].mean()
print(f'Global LGD Assumption : {global_lgd}')

Global LGD Assumption : 10063.590089164785


In [27]:
lgd_by_grade = defaulted_loans.groupby('grade')\
    .agg(
        defaulted_loan_count = ('actual_default_flag','count'),
        avg_realized_lgd = ('realized_lgd','mean'),
        avg_realized_loss = ('realized_loss','mean'),
        avg_ead = ('ead','mean')
    ).reset_index()

lgd_by_grade

,grade,defaulted_loan_count,avg_realized_lgd,avg_realized_loss,avg_ead
0,A,460,7011.446304,7011.446304,13186.793478
1,B,1704,7761.744126,7761.744126,13588.292254
2,C,2924,9378.598331,9378.598331,14892.886457
3,D,1997,10760.132439,10760.132439,16226.952929
4,E,1159,12972.271803,12972.271803,18428.515962
5,F,439,13736.755080,13736.755080,18962.015945
6,G,177,15456.646328,15456.646328,20283.192090


In [36]:
risk_df=risk_df.merge(lgd_by_grade[['grade','avg_realized_lgd']], on='grade', how='left')

risk_df['lgd_estimate'] = risk_df['avg_realized_lgd'].fillna(global_lgd)

risk_df[['grade','realized_lgd', 'lgd_estimate']].tail()

,grade,realized_lgd,lgd_estimate
39995,A,2430.54,7011.446304
39996,D,9424.90,10760.132439
39997,A,0.00,7011.446304
39998,C,2466.45,9378.598331
39999,D,4362.85,10760.132439


## Calculate Expected Loss

#### Expected Loss = PD * LGD * EAD

In [ ]:
risk_df['expected_loss'] = risk_df['pd_score'] * risk_df['lgd_estimate'] * pd_score['ead']

risk_df[['pd_score', 'lgd_estimate', 'ead', 'expected_loss']].head()